# ADC computational pipeline (Jupyter)

Trastuzumab × mc-vc-PAB-MMAE cysteine conjugation.  
Does **not** need PyRosetta. Run this locally or on RunPod after hinge remodel.

If you just finished `runpod/hinge_remodel.ipynb`, set `ANTIBODY_PDB` to `B220_235_rosetta_best.pdb`.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
if (HERE / "config.yaml").exists():
    TROP = HERE
elif (HERE / "examples/trop_adc/config.yaml").exists():
    TROP = HERE / "examples/trop_adc"
else:
    TROP = HERE
    print("WARNING: config.yaml not found in cwd. Set TROP manually.")

ROOT = TROP.parents[1] if (TROP.parents[1] / "packages/adcsim").exists() else TROP.parent.parent
sys.path.insert(0, str(ROOT / "packages"))

print("TROP", TROP)
print("ROOT", ROOT)
print("python", sys.executable)

In [ ]:
from adcsim.config import load_config

cfg_path = TROP / "config.yaml"
cfg = load_config(str(cfg_path))

# After hinge remodel, point this at the FastRelax best PDB (full antibody, not the 22-res fragment).
ANTIBODY_PDB = TROP / "data" / "trastuzumab_boltz_model_0.pdb"
# ANTIBODY_PDB = Path("/workspace/trastuzumab_project/B220_235_rosetta/B220_235_rosetta_best.pdb")

LP_SDF = TROP / "data" / "payload_ensemble_79.sdf"
OUT_DIR = TROP / "results"

cfg["inputs"]["antibody_pdb"] = str(ANTIBODY_PDB)
cfg["inputs"]["lp_sdf"] = str(LP_SDF)
cfg["outputs"]["dir"] = str(OUT_DIR)
cfg["search"]["reuse_library"] = False
cfg["search"]["max_dar"] = 8

print("antibody", ANTIBODY_PDB, "exists", ANTIBODY_PDB.exists())
print("lp_sdf  ", LP_SDF, "exists", LP_SDF.exists())
print("reactive_atom", cfg["chemistry"]["reactive_atom"], cfg["chemistry"]["reactive_atom_name"])
print("sites", cfg["sites"]["pool"])

In [ ]:
import yaml
from pathlib import Path

run_cfg = TROP / "config.run.yaml"
run_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print("wrote", run_cfg)

In [ ]:
from adcsim.pipeline import run

result = run(str(TROP / "config.run.yaml"))
print("highest PASS DAR:", result.get("highest_pass_dar"))
print("ceiling (first empty):", result.get("ceiling"))
print("report:", result.get("report"))

In [ ]:
from IPython.display import Markdown, display
report = Path(result["report"])
display(Markdown(report.read_text()))